# 03 – ETA Prediction

Train and evaluate a Gradient Boosted Regressor to predict the remaining
travel time (ETA) for vessels in the sample dataset.

Because the sample data has no real arrival timestamps, we **synthesise** the
target: for each voyage we calculate the time from the current snapshot to the
last recorded position (a proxy for arrival).


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.data_preprocessing import load_ais_csv, preprocess
from src.feature_engineering import build_features, get_feature_matrix
from src.eta_prediction import ETAPredictor
from src.model_evaluation import regression_metrics, plot_regression_residuals

%matplotlib inline

## 1. Load & Feature-engineer Data

In [ ]:
raw = load_ais_csv('../data/sample/ais_sample.csv')
df = preprocess(raw)
df = build_features(df)
print(df.shape)

## 2. Synthesise ETA Target

In [ ]:
# For each vessel compute the last timestamp as a proxy for arrival
last_ts = df.groupby('mmsi')['timestamp'].transform('max')
df['remaining_seconds'] = (last_ts - df['timestamp']).dt.total_seconds().clip(lower=0)

# Drop the last row of each voyage (remaining_seconds == 0 is trivial)
df_model = df[df['remaining_seconds'] > 0].copy()
print(f'Training rows: {len(df_model)}')

## 3. Train / Test Split

In [ ]:
X = get_feature_matrix(df_model, dropna=True)
y = df_model.loc[X.index, 'remaining_seconds']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')

## 4. Train ETA Model

In [ ]:
model = ETAPredictor(n_estimators=200, learning_rate=0.05, max_depth=4)
model.fit(X_train, y_train)
print('Training complete.')

## 5. Evaluate

In [ ]:
train_metrics = regression_metrics(y_train, model.predict(X_train), prefix='train_')
test_metrics  = regression_metrics(y_test,  model.predict(X_test),  prefix='test_')

metrics_df = pd.DataFrame([train_metrics, test_metrics])
# Convert seconds to hours for readability
for col in [c for c in metrics_df.columns if c.endswith(('mae', 'rmse'))]:
    metrics_df[col] = (metrics_df[col] / 3600).round(3)
metrics_df.index = ['train', 'test']
print('MAE / RMSE shown in hours')
metrics_df

In [ ]:
y_pred_test = model.predict(X_test)
ax = plot_regression_residuals(
    y_test / 3600, y_pred_test / 3600,
    title='ETA Prediction: Predicted vs Actual (hours)',
    xlabel='Actual remaining time (h)',
    ylabel='Predicted remaining time (h)',
)
plt.tight_layout()
plt.show()

## 6. Save Model

In [ ]:
import os
os.makedirs('../models', exist_ok=True)
model.save('../models/eta_model.joblib')
print('Model saved to models/eta_model.joblib')